# Movie Recommendation with RAG: controller, answer and judge agents (Google ADK)

A team of three agents, each with one small job.

| Agent | Job |
|---|---|
| **controller_agent** | the manager: decides the steps and who does them |
| **answer_agent** | searches our movie database and writes the answer |
| **judge_agent** | checks the answer and says `APPROVED` or `REVISE` |

```
 Your question
      |
      v
 CONTROLLER ----> ANSWER agent ----> search_movies_tool (ChromaDB)
      |               |
      |        draft answer
      v               v
 CONTROLLER ----> JUDGE agent  ----> check_movie_tool (is it in our data?)
      |
      |   APPROVED -> return the answer
      |   REVISE   -> ask the answer agent again with the judge's reason
      v
 Final answer
```

The retrieval part is the same as in the single-agent ADK notebook. What is new is the **team**: one agent writes, another agent checks, a third one manages them.

## 0. Setup

In [ ]:
# Run once. (In Colab this takes a couple of minutes.)
%pip install -q google-adk sentence-transformers chromadb

In [ ]:
import os

import chromadb
from sentence_transformers import SentenceTransformer
from google.adk.agents import Agent
from google.adk.tools.agent_tool import AgentTool
from google.adk.runners import InMemoryRunner

# Settings
EMBEDDING_MODEL = "all-MiniLM-L6-v2"     # turns text into 384 numbers
LLM_MODEL = "gemini-2.5-flash"           # the brain of all three agents
TOP_K = 5                                # how many movies the search tool returns

In [ ]:
# Keys are read from the Colab secrets vault (the key icon on the left).
# Outside Colab, or if the secret is missing, we ask you to paste the Gemini key instead.
from getpass import getpass


def load_colab_secret(name):
    """Copy a secret from the Colab vault into the environment. Return True if it worked."""
    try:
        from google.colab import userdata
        os.environ[name] = userdata.get(name)
        return True
    except Exception:
        return False


load_colab_secret("GOOGLE_API_KEY")                # Gemini key: https://aistudio.google.com/apikey
load_colab_secret("HF_TOKEN")                      # optional: speeds up the embedding model download

if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass("Paste your Gemini API key: ")

os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "FALSE"      # use the API key, not Google Cloud

print("Google API key found:", "GOOGLE_API_KEY" in os.environ)
print("HF_TOKEN found      :", "HF_TOKEN" in os.environ)

## 1. Our knowledge base

A small list of 32 movies. Each movie is a Python dictionary.
(Ratings are approximate, and the one-line reviews were written for this demo. They are not real quotes.)

In [ ]:
movies = [
    {"title": "Inception", "year": 2010, "genre": "Sci-Fi, Thriller", "rating": 8.8,
     "cast": "Leonardo DiCaprio, Joseph Gordon-Levitt, Elliot Page",
     "overview": "A thief who steals secrets by entering people's dreams is hired to plant an idea in a target's mind.",
     "review": "Mind-bending and beautifully shot; you will want to watch it twice."},
    {"title": "Interstellar", "year": 2014, "genre": "Sci-Fi, Drama", "rating": 8.7,
     "cast": "Matthew McConaughey, Anne Hathaway, Jessica Chastain",
     "overview": "Astronauts travel through a wormhole to find a new home for humanity as Earth's crops fail.",
     "review": "Epic and emotional, with a score that stays with you."},
    {"title": "The Matrix", "year": 1999, "genre": "Sci-Fi, Action", "rating": 8.7,
     "cast": "Keanu Reeves, Laurence Fishburne, Carrie-Anne Moss",
     "overview": "A hacker discovers that reality is a simulation and joins a rebellion against the machines.",
     "review": "Groundbreaking action and a story that makes you question reality."},
    {"title": "Toy Story", "year": 1995, "genre": "Animation, Family", "rating": 8.3,
     "cast": "Tom Hanks, Tim Allen, Don Rickles",
     "overview": "A cowboy doll feels threatened when a new space ranger toy arrives in a boy's room.",
     "review": "Warm, funny and full of heart for kids and adults alike."},
    {"title": "Finding Nemo", "year": 2003, "genre": "Animation, Family", "rating": 8.2,
     "cast": "Albert Brooks, Ellen DeGeneres, Alexander Gould",
     "overview": "A timid clownfish crosses the ocean to rescue his son.",
     "review": "A colorful, touching adventure about a father's love."},
    {"title": "The Shawshank Redemption", "year": 1994, "genre": "Drama", "rating": 9.3,
     "cast": "Tim Robbins, Morgan Freeman, Bob Gunton",
     "overview": "A banker wrongly jailed for murder finds hope and friendship over decades in prison.",
     "review": "A quiet, powerful story about hope that lifts you up."},
    {"title": "The Godfather", "year": 1972, "genre": "Crime, Drama", "rating": 9.2,
     "cast": "Marlon Brando, Al Pacino, James Caan",
     "overview": "The aging head of a crime family hands control of the empire to his reluctant son.",
     "review": "Slow-burning, masterful storytelling about family and power."},
    {"title": "The Dark Knight", "year": 2008, "genre": "Action, Crime", "rating": 9.0,
     "cast": "Christian Bale, Heath Ledger, Aaron Eckhart",
     "overview": "Batman faces the Joker, a chaotic criminal who pushes Gotham City to its limits.",
     "review": "Dark and intense, with an unforgettable villain."},
    {"title": "Forrest Gump", "year": 1994, "genre": "Drama, Romance", "rating": 8.8,
     "cast": "Tom Hanks, Robin Wright, Gary Sinise",
     "overview": "A kind man with a low IQ unknowingly takes part in major moments of American history.",
     "review": "Gentle, moving and quietly funny; a real feel-good classic."},
    {"title": "Parasite", "year": 2019, "genre": "Thriller, Drama", "rating": 8.5,
     "cast": "Song Kang-ho, Lee Sun-kyun, Cho Yeo-jeong",
     "overview": "A poor family schemes to get jobs in a wealthy household, with unexpected consequences.",
     "review": "Sharp, darkly funny and full of surprises."},
    {"title": "Spirited Away", "year": 2001, "genre": "Animation, Fantasy", "rating": 8.6,
     "cast": "Rumi Hiiragi, Miyu Irino, Mari Natsuki",
     "overview": "A girl wanders into a world of spirits and must work in a bathhouse to free her parents.",
     "review": "Imaginative and magical; every frame feels hand-crafted."},
    {"title": "The Lion King", "year": 1994, "genre": "Animation, Family", "rating": 8.5,
     "cast": "Matthew Broderick, Jeremy Irons, James Earl Jones",
     "overview": "A young lion prince flees after his father's death and later returns to reclaim his kingdom.",
     "review": "A timeless, emotional story with wonderful songs."},
    {"title": "Mad Max: Fury Road", "year": 2015, "genre": "Action, Sci-Fi", "rating": 8.1,
     "cast": "Tom Hardy, Charlize Theron, Nicholas Hoult",
     "overview": "In a desert wasteland, a drifter and a rebel driver flee a tyrant in one long chase.",
     "review": "Non-stop, breathtaking action with very little dialogue."},
    {"title": "La La Land", "year": 2016, "genre": "Romance, Musical", "rating": 8.0,
     "cast": "Ryan Gosling, Emma Stone, John Legend",
     "overview": "A jazz pianist and an aspiring actress fall in love while chasing their dreams in Los Angeles.",
     "review": "Dreamy and romantic with a bittersweet, memorable ending."},
    {"title": "The Grand Budapest Hotel", "year": 2014, "genre": "Comedy, Adventure", "rating": 8.1,
     "cast": "Ralph Fiennes, Tony Revolori, F. Murray Abraham",
     "overview": "A legendary hotel concierge and his young lobby boy are pulled into the theft of a priceless painting.",
     "review": "Quirky, stylish and laugh-out-loud funny."},
    {"title": "Whiplash", "year": 2014, "genre": "Drama, Music", "rating": 8.5,
     "cast": "Miles Teller, J.K. Simmons, Melissa Benoist",
     "overview": "A young drummer is pushed to his limits by a ruthless music teacher.",
     "review": "Tense and electrifying; you will be gripping your seat."},
    {"title": "Get Out", "year": 2017, "genre": "Horror, Thriller", "rating": 7.7,
     "cast": "Daniel Kaluuya, Allison Williams, Bradley Whitford",
     "overview": "A young man visiting his girlfriend's family slowly uncovers a disturbing secret.",
     "review": "Clever, creepy social thriller that keeps you guessing."},
    {"title": "Coco", "year": 2017, "genre": "Animation, Family", "rating": 8.4,
     "cast": "Anthony Gonzalez, Gael Garcia Bernal, Benjamin Bratt",
     "overview": "A boy who dreams of becoming a musician enters the Land of the Dead to uncover his family's history.",
     "review": "Colorful and full of music; keep tissues nearby."},
    {"title": "The Martian", "year": 2015, "genre": "Sci-Fi, Adventure", "rating": 8.0,
     "cast": "Matt Damon, Jessica Chastain, Chiwetel Ejiofor",
     "overview": "An astronaut stranded on Mars uses science and humor to survive until rescue.",
     "review": "Funny, optimistic and a joy for science lovers."},
    {"title": "Arrival", "year": 2016, "genre": "Sci-Fi, Drama", "rating": 7.9,
     "cast": "Amy Adams, Jeremy Renner, Forest Whitaker",
     "overview": "A linguist works to communicate with alien visitors before fear pushes the world toward war.",
     "review": "Thoughtful and moving; a slow-burn sci-fi that rewards patience."},
    {"title": "Back to the Future", "year": 1985, "genre": "Sci-Fi, Comedy", "rating": 8.5,
     "cast": "Michael J. Fox, Christopher Lloyd, Lea Thompson",
     "overview": "A teenager is sent thirty years into the past in a time-traveling car and must make sure his parents fall in love.",
     "review": "Pure fun: clever, fast and endlessly rewatchable."},
    {"title": "Amelie", "year": 2001, "genre": "Romance, Comedy", "rating": 8.3,
     "cast": "Audrey Tautou, Mathieu Kassovitz, Rufus",
     "overview": "A shy Parisian waitress secretly improves the lives of the people around her while searching for love.",
     "review": "Charming, whimsical and uplifting."},
    {"title": "Gladiator", "year": 2000, "genre": "Action, Drama", "rating": 8.5,
     "cast": "Russell Crowe, Joaquin Phoenix, Connie Nielsen",
     "overview": "A betrayed Roman general becomes a gladiator and seeks revenge on the corrupt emperor.",
     "review": "Grand, gritty and emotional with a powerful lead."},
    {"title": "The Silence of the Lambs", "year": 1991, "genre": "Thriller, Crime", "rating": 8.6,
     "cast": "Jodie Foster, Anthony Hopkins, Scott Glenn",
     "overview": "A young FBI trainee asks an imprisoned psychiatrist for help in catching a serial killer.",
     "review": "Chilling and suspenseful with unforgettable performances."},
    {"title": "Up", "year": 2009, "genre": "Animation, Adventure", "rating": 8.3,
     "cast": "Ed Asner, Jordan Nagai, Christopher Plummer",
     "overview": "An elderly widower ties balloons to his house and flies to South America with an unexpected young stowaway.",
     "review": "The opening alone will make you cry; the rest will make you smile."},
    {"title": "Knives Out", "year": 2019, "genre": "Mystery, Comedy", "rating": 7.9,
     "cast": "Daniel Craig, Chris Evans, Ana de Armas",
     "overview": "A detective investigates the death of a wealthy novelist surrounded by a family of suspects.",
     "review": "A witty, twisty whodunit that is great fun to guess along with."},
    {"title": "Everything Everywhere All at Once", "year": 2022, "genre": "Sci-Fi, Comedy", "rating": 7.8,
     "cast": "Michelle Yeoh, Stephanie Hsu, Ke Huy Quan",
     "overview": "A laundromat owner is pulled across parallel universes to stop a threat that could unravel reality.",
     "review": "Wild, inventive and surprisingly touching."},
    {"title": "The Pursuit of Happyness", "year": 2006, "genre": "Drama", "rating": 8.0,
     "cast": "Will Smith, Jaden Smith, Thandiwe Newton",
     "overview": "A struggling salesman and his young son face homelessness while he works toward a career in finance.",
     "review": "Inspiring and heartfelt, anchored by a warm father-son bond."},
    {"title": "Dune", "year": 2021, "genre": "Sci-Fi, Adventure", "rating": 8.0,
     "cast": "Timothee Chalamet, Rebecca Ferguson, Oscar Isaac",
     "overview": "A noble young heir travels to a dangerous desert planet to protect his family and its people.",
     "review": "Visually stunning and immersive on a big screen."},
    {"title": "Paddington 2", "year": 2017, "genre": "Family, Comedy", "rating": 7.8,
     "cast": "Ben Whishaw, Hugh Bonneville, Sally Hawkins",
     "overview": "A polite bear is wrongly jailed for a theft and his adoptive family works to prove his innocence.",
     "review": "Sweet, funny and kind-hearted; pure comfort viewing."},
    {"title": "Your Name", "year": 2016, "genre": "Animation, Romance", "rating": 8.4,
     "cast": "Ryunosuke Kamiki, Mone Kamishiraishi, Ryo Narita",
     "overview": "Two teenagers mysteriously swap bodies and try to find each other across distance and time.",
     "review": "Gorgeous animation and a romance that hits hard."},
    {"title": "Oppenheimer", "year": 2023, "genre": "Drama, History", "rating": 8.3,
     "cast": "Cillian Murphy, Emily Blunt, Matt Damon",
     "overview": "The story of the physicist who led the creation of the first atomic bomb and of its aftermath.",
     "review": "Dense, intense and gripping; a big, serious drama."},
]

print("Movies in our knowledge base:", len(movies))
movies[0]

## 2. Turn each movie into text

The embedding model reads text, so we describe every movie as one string: title, year, genre, rating, cast, plot and review. Everything we want to be searchable goes in.
The same string is also what the agent reads when the search tool returns a movie.

In [ ]:
def movie_to_text(movie):
    """Describe one movie as a single line of text."""
    return (f"{movie['title']} ({movie['year']}) | Genre: {movie['genre']} | Rating: {movie['rating']} "
            f"| Cast: {movie['cast']} | Plot: {movie['overview']} | Review: {movie['review']}")


movie_texts = []
for movie in movies:
    movie_texts.append(movie_to_text(movie))

print(movie_texts[3])

## 3. Embeddings and the ChromaDB vector store

An **embedding** is a list of numbers that captures the *meaning* of a text. Texts with similar meaning get similar numbers, so "a funny cartoon for kids" ends up close to *Toy Story*, even though they share no words.

**ChromaDB** is a vector database. It stores the vectors together with each movie's details and quickly finds the closest ones.
The first run downloads the embedding model (about 90 MB).

In [ ]:
embedder = SentenceTransformer(EMBEDDING_MODEL)
movie_vectors = embedder.encode(movie_texts)

print("Shape:", movie_vectors.shape)                       # (32 movies, 384 numbers each)
print("First 5 numbers of movie 0:", movie_vectors[0][:5])

In [ ]:
def build_collection(movies, vectors, texts):
    """Create an in-memory ChromaDB collection holding our movies and their vectors."""
    client = chromadb.Client()
    collection = client.get_or_create_collection(name="movie_collection")

    collection.add(
        ids=[str(number) for number in range(len(movies))],
        embeddings=vectors.tolist(),        # we computed the vectors ourselves
        documents=texts,
        metadatas=movies,                   # each movie dictionary is stored next to its vector
    )
    return collection


collection = build_collection(movies, movie_vectors, movie_texts)
print("Movies stored in ChromaDB:", collection.count())

## 4. Retrieval function

Embed the question, then ask ChromaDB for the closest movies. A **smaller distance means a closer match**.

In [ ]:
def search_movies_chroma(question, k=TOP_K):
    """Return the k movies whose meaning is closest to the question."""
    question_vector = embedder.encode([question]).tolist()
    results = collection.query(query_embeddings=question_vector, n_results=k)

    found = []
    for movie, distance in zip(results["metadatas"][0], results["distances"][0]):
        movie = dict(movie)
        movie["distance"] = round(float(distance), 3)
        found.append(movie)
    return found


def show_movies(found):
    """Print the movies as simple info cards."""
    for movie in found:
        print(f"{movie['title']} ({movie['year']})  |  Rating: {movie['rating']}  |  {movie['genre']}  |  distance {movie['distance']}")
        print(f"   Cast:    {movie['cast']}")
        print(f"   Summary: {movie['overview']}")
        print(f"   Review:  {movie['review']}")
        print()


show_movies(search_movies_chroma("a funny animated movie the whole family can enjoy", 3))

## 5. Two tools

Each worker agent gets **one tool**, matching its job.

- `search_movies_tool` is for the **answer agent**. It finds movies that match the question.
- `check_movie_tool` is for the **judge agent**. It looks a title up by name in our database, so the judge can catch a movie we do not have. This is a plain lookup, not a vector search.

As before, a tool is a normal Python function. Its **docstring** tells the agent when to use it.

In [ ]:
last_found = []      # the movies from the most recent search


def search_movies_tool(question: str) -> str:
    """Search the movie database. Use this to find movies by genre, cast, plot, mood or review.

    Args:
        question: What the user is looking for, for example "animated family movies".

    Returns:
        A text list of the best matching movies.
    """
    global last_found
    last_found = search_movies_chroma(question)

    result = ""
    for movie in last_found:
        result = result + "- " + movie_to_text(movie) + "\n"
    return result


def check_movie_tool(title: str) -> str:
    """Check whether a movie title exists in our database.

    Args:
        title: The exact movie title, for example "Toy Story".

    Returns:
        The movie's details, or NOT IN DATABASE.
    """
    for movie in movies:
        if movie["title"].lower() == title.lower():
            return movie_to_text(movie)
    return "NOT IN DATABASE"


# Tools are just functions, so we can test them without any agent.
print(search_movies_tool("space adventure")[:300])
print(check_movie_tool("Toy Story")[:100])
print(check_movie_tool("Titanic"))

## 6. Create the three agents

All three are normal ADK `Agent`s. Two things are new:

- **`description`**: a one-line summary of what the agent does. The controller reads it to know who to call.
- **`AgentTool`**: wraps an agent so that *another agent can call it like a tool*. This is how the controller manages the other two.

The controller has **no movie tools of its own**. It only calls the other agents, so its `instruction` is the plan of the whole team.

In [ ]:
answer_agent = Agent(
    name="answer_agent",
    model=LLM_MODEL,
    description="Answers a movie question using our movie database. Send it the user's question, and any feedback to fix.",
    instruction=(
        "You are a friendly movie assistant. "
        "ALWAYS use the search_movies_tool before answering. "
        "Answer using ONLY the movies the tool returns. "
        "If the user wants recommendations, pick the best 3 and give one short reason for each. "
        "If the request includes feedback from a reviewer, fix the problems it mentions."
    ),
    tools=[search_movies_tool],
)

judge_agent = Agent(
    name="judge_agent",
    model=LLM_MODEL,
    description="Checks a draft answer. Send it the question and the draft answer. It replies APPROVED or REVISE with a reason.",
    instruction=(
        "You are a strict reviewer. You receive a user's question and a draft answer. "
        "For every movie the draft recommends, call check_movie_tool with its title. "
        "Reply 'REVISE: <short reason>' if a movie is NOT IN DATABASE, or if a movie clearly does not fit the question "
        "(for example the wrong actor or the wrong genre). "
        "Otherwise reply 'APPROVED'. Reply with one line only."
    ),
    tools=[check_movie_tool],
)

controller_agent = Agent(
    name="controller_agent",
    model=LLM_MODEL,
    instruction=(
        "You manage a small movie team. Follow these steps. "
        "1. If the user's message is not about movies (for example a greeting), reply politely yourself and stop. "
        "2. Otherwise call answer_agent with the user's question. "
        "3. Call judge_agent with the question and the draft answer. "
        "4. If the judge says APPROVED, reply with the draft answer exactly as written. "
        "5. If the judge says REVISE, call answer_agent again with the question and the judge's reason, "
        "then reply with the new answer. Do this at most once. "
        "Never write movie recommendations yourself."
    ),
    tools=[AgentTool(agent=answer_agent), AgentTool(agent=judge_agent)],
)

print("Team ready:", controller_agent.name, "->", answer_agent.name, "+", judge_agent.name)

## 7. Run the team

We talk only to the **controller**. It calls the other two agents behind the scenes. As before, the runner keeps a session, and `await` works directly in a notebook cell.

In [ ]:
runner = InMemoryRunner(agent=controller_agent)


async def ask_agent(question):
    """Send a question to the controller and return its final answer as text."""
    events = await runner.run_debug(question, quiet=True)

    answer = ""
    for event in events:
        if event.is_final_response() and event.content and event.content.parts:
            answer = event.content.parts[0].text
    return answer

In [ ]:
question = "Something like Interstellar, but shorter and more hopeful"

answer = await ask_agent(question)

print("Question:", question)
print()
print("Movies the answer agent found:")
show_movies(last_found)
print("Final answer:")
print(answer)

**Your turn:** change the question and run it again. Try a genre, an actor, a mood, or just say hello.

In [ ]:
answer = await ask_agent("Movies with Tom Hanks")

show_movies(last_found)
print(answer)

### Watch the team work

Print the controller's events to see the plan in action: it calls the answer agent, then the judge, and maybe the answer agent again. This is the part a single agent did not have.

In [ ]:
events = await runner.run_debug("Recommend a romantic movie", quiet=True)

for event in events:
    for part in event.content.parts:
        if part.function_call:
            print("CONTROLLER CALLS", part.function_call.name, "->", str(part.function_call.args)[:150])
        elif part.function_response:
            print("   REPLY FROM", part.function_response.name, "->", str(part.function_response.response)[:200])
        elif part.text:
            print(event.author.upper() + ":", part.text[:300])

## One agent vs a team

| | Single ADK agent | Controller + answer + judge |
|---|---|---|
| Who writes the answer? | the one agent | the answer agent |
| Who checks the answer? | nobody | the judge agent |
| Who plans the steps? | the agent, from its instruction | the controller, from its instruction |
| Can a bad answer be fixed? | no | yes, the controller asks for a revision |
| Cost | 2 or more LLM calls | 5 or more LLM calls |

## What to try next

- **Break the judge on purpose:** change the answer agent's instruction to "recommend any famous movie" and watch the judge reply `REVISE` for movies that are not in our data.
- **Make the judge stricter:** add a rule such as "reject answers that give fewer than 3 movies".
- **Ask something that is not about movies** (for example *"hello"*). Does the controller skip the other two agents?
- **Allow two revisions:** change step 5 of the controller's instruction. What happens to the cost?
- **Add the poster cards and the Gradio app** from the single-agent ADK notebook. They work unchanged, because `last_found` is still filled by the search tool.